In [ ]:
%python
config_id           = dbutils.widgets.get("config_id")
bronze_table        = dbutils.widgets.get("bronze_table")
source_schema       = dbutils.widgets.get("source_schema")
source_object_name  = dbutils.widgets.get("source_object_name")
load_type           = dbutils.widgets.get("load_type")
primary_key_cols    = dbutils.widgets.get("primary_key_cols")

# Only needed to write this run's own SILVER row to the audit table —
# all values are passed in from IngestionOrchestrator, no config/audit re-query.
source_name         = dbutils.widgets.get("source_name")
pipeline_name       = dbutils.widgets.get("pipeline_name")
target_schema       = dbutils.widgets.get("target_schema")
target_table        = dbutils.widgets.get("target_table")
audit_table         = dbutils.widgets.get("audit_table")
department_id       = int(dbutils.widgets.get("department_id") or "0")
job_run_id          = dbutils.widgets.get("job_run_id")
config_master_id    = dbutils.widgets.get("config_master_id")
frequency           = dbutils.widgets.get("frequency") or None

print(f"✅ Silver running for config_id={config_id} bronze_table={bronze_table}")

In [ ]:
%python
# dbutils.notebook.run() executes this notebook in its own REPL — sys.path from
# the calling notebook (main.py) does not carry over, so it's set again here.
import sys
sys.path.append("..")

from types import SimpleNamespace

from ingestion.utils.audit import AuditLogger
from ingestion.utils.config_manager import AUDIT_STATUS_SUCCESS

# AuditLogger.start_run() only reads a handful of attributes off `task`/`source_sys`
# via dot access — build lightweight stand-ins instead of a full IngestionTaskConfig
# so this notebook never has to re-query config_master/ingestion_config.
task_ns = SimpleNamespace(
    config_id=int(config_id),
    effective_delta_layer="SILVER",
    load_type=load_type,
    frequency=frequency,
    source_schema=source_schema,
    source_object_name=source_object_name,
    target_schema=target_schema,
    target_table=target_table,
)
source_sys_ns = SimpleNamespace(source_name=source_name)

audit = AuditLogger(spark, audit_table=audit_table, department_id=department_id)
audit_run = audit.start_run(
    task=task_ns,
    source_sys=source_sys_ns,
    job_context={"job_run_id": job_run_id},
    pipeline_name=pipeline_name,
    config_master_id=int(config_master_id) if config_master_id else None,
)

In [ ]:
%python
try:
    # TODO: real Bronze → Silver transform goes here (read `bronze_table`,
    # dedupe/merge on `primary_key_cols`, write to the Silver target).
    rows_read = 0

    audit.complete_run(
        audit_run=audit_run,
        status=AUDIT_STATUS_SUCCESS,
        rows_read=rows_read,
        rows_copied=rows_read,
    )
    print(f"✅ Silver SUCCESS for config_id={config_id} bronze_table={bronze_table}")
except Exception as exc:
    audit.fail_run(audit_run=audit_run, error_code=type(exc).__name__, error_message=str(exc))
    raise